# Synthetic tabular environment

`SyntheticEnv-v1` is a custom environment bundled with mouse-env. It generates a **random finite discrete MDP** — a fully specified tabular RL problem with configurable state and action spaces. By default, each env instance keeps its generated MDP across resets; pass `episode_reset_options={"regenerate_map": True}` on `EnvConfig` to sample a fresh MDP every episode reset.

## Why use a synthetic env?

Real Gymnasium environments like CartPole or MountainCar have fixed structure that may not match what you want to test. `SyntheticEnv-v1` is configurable and reproducible via `kwargs={"map_seed": ...}`. Use it when:

- You want to test a tabular RL algorithm without committing to a specific env topology.
- You need ground-truth Q* values to compare against a learned policy (the `env_q_star` provider computes exact Q-values via value iteration).
- You want to sweep over state/action space sizes to understand how an algorithm scales.

## `env_q_star`: env-computed Q*

Some environments can compute Q* themselves because they have full access to the transition model. `SyntheticEnv-v1` and `Procedural-FrozenLake-v1` both support this via `q_star_source={"provider": "env_q_star"}`.

When this provider is active, the env runs value iteration internally whenever the MDP changes (on construction, or when a new random map is generated). The resulting Q-table is injected into every step output as `outputs[i]["info_q_star"]` — no external file, no separate compute step.

Constructor options (passed via `kwargs`):

| Option | Purpose |
|--------|---------|
| `obs_size` | Number of states in the MDP |
| `action_size` | Number of actions per state |

Episode reset options (passed via `EnvConfig.episode_reset_options`):

| Option | Purpose |
|--------|---------|
| `regenerate_map` | Sample a fresh random MDP on every reset (requires no fixed `map`) |

In [ ]:
from mouse_envs import EnvConfig, make_env, make_group_env

## Configure a random MDP

In [ ]:
cfgs = [
    EnvConfig(
        id="SyntheticEnv-v1",
        reset_seed=i,
        episodes_per_task=10,
        kwargs={"obs_size": 8, "action_size": 3, "max_episode_steps": 50},
        q_star_source={"provider": "env_q_star"},
    )
    for i in range(2)
]
env = make_group_env(cfgs)

## Inspect the stream

`SyntheticEnv-v1` emits `info_map` by default (`emit_map=True`). The first step after construction includes `outputs[i]["info_map"]` — a JSON string encoding the full MDP: transition matrix, goal flags, rewards, and start-state distribution. Subsequent steps do not carry `info_map` unless `episode_reset_options={"regenerate_map": True}`, in which case each reset frame carries the newly sampled MDP.

In [ ]:
import json

outputs = env.step(env.sample_random_input())

for i, r in enumerate(outputs):
    print(
        f"env={i}  obs={r['observation'].tolist()}  "
        f"q_star_shape={r['info_q_star'].shape}  "
        f"has_map={'info_map' in r}"
    )

## Inspect the map

The `map` JSON payload contains:
- `transition` — `[S, A]` next-state indices
- `goal` — `[S, A]` bool flags (True when the transition is terminal/goal-reaching)
- `reward` — `[S, A]` per-(state, action) rewards
- `start` — `[S]` bool flags marking valid start states

In [ ]:
import numpy as np

# Map is emitted on the first step; capture it from env 0.
map_data = json.loads(outputs[0]["info_map"])
print("Map keys:", list(map_data.keys()))

transition = np.array(map_data["transition"])  # [S, A] next-state indices
goal       = np.array(map_data["goal"])         # [S, A] bool
reward     = np.array(map_data["reward"])        # [S, A]
start      = np.array(map_data["start"])         # [S] bool

n_states, n_actions = transition.shape
print(f"States: {n_states}, Actions: {n_actions}")
print(f"Start states: {np.where(start)[0].tolist()}")
print(f"Goal (state, action) pairs: {list(zip(*np.where(goal)))}")
print()
print("Next-state table (states × actions):")
print(transition)
print()
print("Reward table (states × actions):")
print(np.round(reward, 3))

env.close()